In [ ]:
import cv2

# Load OpenCV's pre-trained face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("Face Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2

# Load Haar cascades for face and nose
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
nose_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_mcs_nose.xml")

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        roi_gray = gray[y:y + h, x:x + w]
        roi_color = frame[y:y + h, x:x + w]

        noses = nose_cascade.detectMultiScale(roi_gray, 1.3, 5)
        for (nx, ny, nw, nh) in noses:
            cv2.rectangle(roi_color, (nx, ny), (nx + nw, ny + nh), (0, 0, 255), 2)
            nose_center_x = nx + nw // 2

            # Head direction logic based on nose x position relative to face
            if nose_center_x < w / 3:
                direction = "LOOKING LEFT"
            elif nose_center_x > 2 * w / 3:
                direction = "LOOKING RIGHT"
            else:
                direction = "LOOKING FORWARD"

            cv2.putText(frame, direction, (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            break  # only use the first detected nose

    cv2.imshow("Head Direction", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
from retinaface import RetinaFace

cap = cv2.VideoCapture(0)

total_frames = 0
attentive_frames = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    total_frames += 1

    # Detect faces using RetinaFace
    detections = RetinaFace.detect_faces(frame)

    current_status = "NOT ATTENTIVE"

    if isinstance(detections, dict):
        for key in detections:
            identity = detections[key]
            facial_area = identity["facial_area"]
            landmarks = identity["landmarks"]

            x1, y1, x2, y2 = facial_area

            # Draw face box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Draw eye landmarks
            left_eye = landmarks["left_eye"]
            right_eye = landmarks["right_eye"]

            cv2.circle(frame, left_eye, 5, (255, 0, 0), -1)
            cv2.circle(frame, right_eye, 5, (255, 0, 0), -1)

            # Engagement logic:
            # If both eyes detected → attentive
            if left_eye and right_eye:
                current_status = "ATTENTIVE"
                attentive_frames += 1

            break  # only first face

    # Calculate attention %
    attention_percent = (attentive_frames / total_frames) * 100

    cv2.putText(frame, f"Status: {current_status}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.putText(frame, f"Attention: {attention_percent:.2f}%", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    cv2.imshow("Phase 3 - RetinaFace Engagement", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2

# Load cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

cap = cv2.VideoCapture(0)

total_frames = 0
attentive_frames = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    total_frames += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    current_status = "NOT ATTENTIVE"

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 5)

        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)

        if len(eyes) >= 2:
            current_status = "ATTENTIVE"
            attentive_frames += 1

        break  # only consider first detected face

    # Calculate attention percentage
    if total_frames > 0:
        attention_percent = (attentive_frames / total_frames) * 100
    else:
        attention_percent = 0

    # Display status
    cv2.putText(frame, f"Status: {current_status}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.putText(frame, f"Attention: {attention_percent:.2f}%", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    cv2.imshow("Phase 4 - Attention Analytics", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()